In [1]:
# Cell 1: Imports, Device & Directory Configurations
import os
import random
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models
from torchvision.models import swin_v2_t, vit_b_16

from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, f1_score,
    precision_score, recall_score, confusion_matrix, classification_report
)

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", DEVICE)

# Dataset Root Paths
DATASET_FOLDER = os.path.join("archive", "nih_imagefolder")
DATA_ROOT = os.environ.get("PNEUMONIA_DATA_ROOT", os.path.join(os.getcwd(), DATASET_FOLDER))

# Saved Checkpoint Paths for all 5 Models
RESNET_PATH = "./updated1_output/best_s2_mixup.pth"
DENSENET_PATH = "./e2_output/best_s2_mixup.pth"
EFFNET_PATH = "./ef_output/best_s2_mixup.pth"
SWIN_PATH = "./swin_output/best_s2_mixup.pth"
VIT_PATH = "./vit_output/best_s2_mixup.pth"

OUTPUT_DIR = "./ensemble_models/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH = 16
NUM_WORKERS = 4
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("Environment and Seeds Configured.")

Using Device: cpu
Environment and Seeds Configured.


In [2]:
# Cell 2: Data Loading & Preprocessing
# Standard resolution 256x256 is kept for Swin Transformer compatibility.
# For 224x224 models (ResNet, DenseNet, EfficientNet, ViT), dynamic bilinear interpolation is applied.
IMG_SIZE_SWIN = 256

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_SWIN, IMG_SIZE_SWIN)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_dir  = os.path.join(DATA_ROOT, "val")
test_dir = os.path.join(DATA_ROOT, "test")

missing_splits = [split for split in ("val", "test") if not os.path.isdir(os.path.join(DATA_ROOT, split))]
if missing_splits:
    raise FileNotFoundError(
        f"Prepared NIH dataset not found at {DATA_ROOT}. Run prepare_nih_xray14.py first. "
        f"Missing: {', '.join(missing_splits)}."
    )

val_ds  = datasets.ImageFolder(val_dir, transform=val_transform)
test_ds = datasets.ImageFolder(test_dir, transform=val_transform)

# DataLoaders: Validation set for candidate evaluation/tuning; Test set strictly for final one-time evaluation
val_loader  = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("Class mapping:", val_ds.class_to_idx)
# Index 0: NORMAL, Index 1: PNEUMONIA
print(f"Validation set size (used for weight search & selection): {len(val_ds)}")
print(f"Test set size (strictly held out for final reporting): {len(test_ds)}")


Class mapping: {'NORMAL': 0, 'PNEUMONIA': 1}
Validation set size (used for weight search & selection): 10322
Test set size (strictly held out for final reporting): 10416


In [3]:
# Cell 3: Load All 5 Base Models (ResNet50, DenseNet121, EfficientNet-B3, Swin-T, ViT-B/16)
def load_resnet():
    model = models.resnet50(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, 1)
    model.load_state_dict(torch.load(RESNET_PATH, map_location=DEVICE))
    return model.eval().to(DEVICE)

def load_densenet():
    model = models.densenet121(pretrained=False)
    model.classifier = nn.Linear(model.classifier.in_features, 1)
    model.load_state_dict(torch.load(DENSENET_PATH, map_location=DEVICE))
    return model.eval().to(DEVICE)

def load_effnet():
    model = models.efficientnet_b3(pretrained=False)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
    model.load_state_dict(torch.load(EFFNET_PATH, map_location=DEVICE))
    return model.eval().to(DEVICE)

def load_swin():
    model = swin_v2_t(weights=None)
    model.head = nn.Linear(model.head.in_features, 1)
    model.load_state_dict(torch.load(SWIN_PATH, map_location=DEVICE))
    return model.eval().to(DEVICE)

def load_vit():
    model = vit_b_16(weights=None)
    model.heads.head = nn.Linear(model.heads.head.in_features, 1)
    model.load_state_dict(torch.load(VIT_PATH, map_location=DEVICE))
    return model.eval().to(DEVICE)

print("Loading all 5 models...")
models_dict = {
    "ResNet": load_resnet(),
    "DenseNet": load_densenet(),
    "EfficientNet": load_effnet(),
    "Swin": load_swin(),
    "ViT": load_vit()
}

# Explicitly freeze all parameters across all 5 models
for m_name, model in models_dict.items():
    for param in model.parameters():
        param.requires_grad = False
    print(f"Loaded and frozen: {m_name}")

MODEL_NAMES = list(models_dict.keys())
print("All 5 base models successfully loaded and ready.")


Loading all 5 models...


C:\Users\CSE\AppData\Roaming\Python\Python314\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\CSE\AppData\Roaming\Python\Python314\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


FileNotFoundError: [Errno 2] No such file or directory: './updated1_output/best_s2_mixup.pth'

In [ ]:
# Cell 4: Prediction Extraction Function
def get_model_predictions(models_dict, loader, device):
    """
    Extracts binary pneumonia probabilities (after sigmoid) for all 5 models.
    Inputs are resized dynamically: 256x256 for Swin, 224x224 for ResNet, DenseNet, EfficientNet, and ViT.
    Returns:
        prob_matrix: np.ndarray of shape (N, 5) with predicted pneumonia probabilities.
        labels: np.ndarray of shape (N,) with ground-truth binary labels.
    """
    all_probs = {name: [] for name in models_dict.keys()}
    all_labels = []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Extracting Model Predictions"):
            imgs = imgs.to(device)
            # CNNs and ViT take 224x224
            imgs_224 = F.interpolate(imgs, size=(224, 224), mode='bilinear', align_corners=False)

            # ResNet50
            logit_res = models_dict["ResNet"](imgs_224)
            prob_res = torch.sigmoid(logit_res).cpu().numpy().flatten()
            all_probs["ResNet"].extend(prob_res)

            # DenseNet121
            logit_den = models_dict["DenseNet"](imgs_224)
            prob_den = torch.sigmoid(logit_den).cpu().numpy().flatten()
            all_probs["DenseNet"].extend(prob_den)

            # EfficientNet-B3
            logit_eff = models_dict["EfficientNet"](imgs_224)
            prob_eff = torch.sigmoid(logit_eff).cpu().numpy().flatten()
            all_probs["EfficientNet"].extend(prob_eff)

            # Swin Transformer (256x256)
            logit_swin = models_dict["Swin"](imgs)
            prob_swin = torch.sigmoid(logit_swin).cpu().numpy().flatten()
            all_probs["Swin"].extend(prob_swin)

            # Vision Transformer (ViT-B/16, 224x224)
            logit_vit = models_dict["ViT"](imgs_224)
            prob_vit = torch.sigmoid(logit_vit).cpu().numpy().flatten()
            all_probs["ViT"].extend(prob_vit)

            all_labels.extend(labels.numpy().flatten())

    prob_matrix = np.column_stack([all_probs[name] for name in MODEL_NAMES])
    labels_arr = np.array(all_labels)

    # Verification: probabilities in [0, 1]
    assert np.all((prob_matrix >= 0.0) & (prob_matrix <= 1.0)), "Error: Probabilities out of range [0, 1]!"
    return prob_matrix, labels_arr

print("Extracting predictions on Validation Set...")
val_probs, val_labels = get_model_predictions(models_dict, val_loader, DEVICE)

print("Extracting predictions on Test Set...")
test_probs, test_labels = get_model_predictions(models_dict, test_loader, DEVICE)

print(f"Validation probability matrix shape: {val_probs.shape}")
print(f"Test probability matrix shape: {test_probs.shape}")


In [ ]:
# Cell 5: Systematic Candidate Weight Generation
def generate_candidate_weights(n_models=5, step=0.05):
    """
    Systematically generates candidate weight combinations where:
      - w_i >= 0 for each model
      - sum(w) == 1.0 (verified within 1e-6)
    Includes:
      - Equal weighting baseline [0.2, 0.2, 0.2, 0.2, 0.2]
      - Single-model dominant combinations (e.g. 0.6 on ViT or Swin)
      - CNN-focused, Transformer-focused, and mixed hybrid candidate combinations
      - Grid simplex combinations with discretization step.
    """
    candidates = []
    
    # Baseline equal weighting
    candidates.append([1.0 / n_models] * n_models)

    # Individual model dominance combinations (primary model at 0.50 or 0.60, rest shared)
    for i in range(n_models):
        for lead_w in [0.40, 0.50, 0.60]:
            rem = round(1.0 - lead_w, 4)
            w = [round(rem / (n_models - 1), 4)] * n_models
            w[i] = lead_w
            # Adjust minor floating rounding
            diff = 1.0 - sum(w)
            w[0] += diff
            candidates.append([round(x, 4) for x in w])

    # CNN-dominant vs Transformer-dominant combinations
    # ResNet(0), DenseNet(1), EffNet(2), Swin(3), ViT(4)
    candidates.append([0.25, 0.25, 0.25, 0.125, 0.125]) # CNN heavy
    candidates.append([0.10, 0.10, 0.10, 0.35, 0.35])   # Transformer heavy
    candidates.append([0.15, 0.15, 0.20, 0.25, 0.25])   # Balanced hybrid
    candidates.append([0.10, 0.20, 0.30, 0.20, 0.20])   # Dense/EffNet priority
    candidates.append([0.15, 0.15, 0.30, 0.20, 0.20])   # EffNet priority
    candidates.append([0.10, 0.15, 0.25, 0.25, 0.25])   # Multi-lead

    # Controlled simplex grid generation
    # Multiples of 0.10 to maintain clean search space without combinatorial explosion
    steps = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
    for comb in itertools.product(steps, repeat=n_models):
        if abs(sum(comb) - 1.0) < 1e-6:
            candidates.append([round(x, 4) for x in comb])

    # Remove duplicates
    unique_candidates = []
    seen = set()
    for w in candidates:
        w_tuple = tuple(np.round(w, 4))
        if w_tuple not in seen:
            seen.add(w_tuple)
            unique_candidates.append(np.array(w_tuple, dtype=np.float64))

    # Assert validity
    for w in unique_candidates:
        assert np.all(w >= -1e-6), f"Negative weight detected: {w}"
        assert abs(np.sum(w) - 1.0) < 1e-5, f"Weights do not sum to 1: {w}, sum={np.sum(w)}"

    return np.array(unique_candidates)

candidate_weights = generate_candidate_weights(n_models=5)
print(f"Generated {len(candidate_weights)} valid candidate weight combinations.")
print("Sample Candidate 1 (Equal):", candidate_weights[0])
print("Sample Candidate 2:", candidate_weights[1])
print("Sample Candidate 3:", candidate_weights[2])


In [ ]:
# Cell 6: Validation Weight Evaluation & Selection of Best Fixed Weights
def evaluate_predictions(y_true, y_probs):
    """Calculates Accuracy, Precision, Recall, F1 (optimal threshold), and ROC-AUC."""
    auc = roc_auc_score(y_true, y_probs)
    
    # Search threshold for primary metric: F1-score
    best_thr, best_f1, best_acc, best_prec, best_rec = 0.5, 0.0, 0.0, 0.0, 0.0
    for thr in np.linspace(0.1, 0.9, 81):
        preds = (y_probs >= thr).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr
            best_acc = accuracy_score(y_true, preds)
            best_prec = precision_score(y_true, preds, zero_division=0)
            best_rec = recall_score(y_true, preds, zero_division=0)
            
    return {
        "AUC": auc, "Accuracy": best_acc, "Precision": best_prec,
        "Recall": best_rec, "F1": best_f1, "Threshold": best_thr
    }

def evaluate_all_candidate_weights(candidate_weights, probs_matrix, y_true):
    """
    Evaluates every candidate weight vector on the given dataset split.
    """
    results = []
    for idx, w in enumerate(candidate_weights):
        # Weighted ensemble probability: P = sum(w_i * P_i)
        combined_prob = np.dot(probs_matrix, w)
        metrics = evaluate_predictions(y_true, combined_prob)
        results.append({
            "Combination": idx + 1,
            "ResNet": w[0],
            "DenseNet": w[1],
            "EfficientNet": w[2],
            "Swin": w[3],
            "ViT": w[4],
            "Accuracy": metrics["Accuracy"],
            "Precision": metrics["Precision"],
            "Recall": metrics["Recall"],
            "F1": metrics["F1"],
            "AUC": metrics["AUC"],
            "Threshold": metrics["Threshold"]
        })
    return pd.DataFrame(results)

print("--- Evaluating All Candidate Weights on VALIDATION Set (Zero Leakage) ---")
val_weight_results = evaluate_all_candidate_weights(candidate_weights, val_probs, val_labels)

# Sort by primary metric: F1-score
val_weight_results = val_weight_results.sort_values(by=["F1", "AUC", "Accuracy"], ascending=False).reset_index(drop=True)

print("\nTop 10 Candidate Weight Combinations on Validation Set:")
print(val_weight_results.head(10).to_string(index=False))

best_val_row = val_weight_results.iloc[0]
best_fixed_weights = np.array([
    best_val_row["ResNet"], best_val_row["DenseNet"], best_val_row["EfficientNet"],
    best_val_row["Swin"], best_val_row["ViT"]
])

# Assert validity
assert abs(np.sum(best_fixed_weights) - 1.0) < 1e-5, "Best weights do not sum to 1"
assert np.all(best_fixed_weights >= 0), "Negative weight in best weights"

print("\n" + "="*60)
print(f"BEST FIXED VALIDATION WEIGHT COMBINATION (Combination #{int(best_val_row['Combination'])}):")
print(f"ResNet={best_fixed_weights[0]:.4f}, DenseNet={best_fixed_weights[1]:.4f}, "
      f"EffNet={best_fixed_weights[2]:.4f}, Swin={best_fixed_weights[3]:.4f}, ViT={best_fixed_weights[4]:.4f}")
print(f"Val F1: {best_val_row['F1']:.4f} | Val AUC: {best_val_row['AUC']:.4f} | "
      f"Val Acc: {best_val_row['Accuracy']*100:.2f}% | Val Precision: {best_val_row['Precision']:.4f} | Val Recall: {best_val_row['Recall']:.4f}")
print("="*60)


In [ ]:
# Cell 7: Input-Dependent Dynamic Weight Selection Mechanism
"""
Dynamic Weighting Strategy:
For each individual X-ray image:
  1. Calculate each model's prediction confidence: c_m = |P_m - 0.5| * 2  (0 = maximally uncertain, 1 = completely confident).
  2. For each candidate weight vector w in candidate_weights:
     - Compute confidence alignment: Score_conf(w) = sum_m (w_m * c_m)  [favors candidates that place heavier weights on confident models].
     - Modulate with validation quality prior: Score_prior(w) = Val_F1(w).
     - Combined Suitability: Score(w) = Score_conf(w) * (Score_prior(w) ** 2).
  3. Select the candidate weight vector with highest score for this specific image.
  4. Verify:
     - All w_i >= 0
     - sum(w) == 1.0
"""

# Cache validation F1 prior for each candidate
candidate_val_f1 = dict(zip(val_weight_results["Combination"], val_weight_results["F1"]))

def select_dynamic_weights_for_sample(model_probs, candidates, candidate_f1_scores):
    """
    Selects optimal weight vector for a single X-ray image based on model confidences.
    model_probs: array of shape (5,)
    """
    # Model confidence: distance from ambiguous decision boundary 0.5
    confidences = np.abs(model_probs - 0.5) * 2.0  # shape (5,)

    best_score = -1.0
    best_w = None

    for idx, w in enumerate(candidates):
        conf_align = np.dot(w, confidences)
        prior = candidate_f1_scores.get(idx + 1, 0.5)
        total_score = conf_align * (prior ** 2)

        if total_score > best_score:
            best_score = total_score
            best_w = w

    assert abs(np.sum(best_w) - 1.0) < 1e-5, f"Selected dynamic weights do not sum to 1: {best_w}"
    assert np.all(best_w >= 0), f"Negative weight in dynamic weights: {best_w}"
    return best_w

def dynamic_ensemble_predict(probs_matrix, candidates, candidate_f1_scores):
    """
    Applies input-dependent weight selection across all samples in the dataset.
    Returns:
        final_probs: np.ndarray of shape (N,)
        selected_weights_matrix: np.ndarray of shape (N, 5)
    """
    N = probs_matrix.shape[0]
    final_probs = np.zeros(N)
    selected_weights = np.zeros((N, 5))

    for i in range(N):
        w = select_dynamic_weights_for_sample(probs_matrix[i], candidates, candidate_f1_scores)
        selected_weights[i] = w
        final_probs[i] = np.dot(w, probs_matrix[i])

    return final_probs, selected_weights

print("Running Dynamic Weight Selection on Validation Set...")
val_dyn_probs, val_dyn_weights = dynamic_ensemble_predict(val_probs, candidate_weights, candidate_val_f1)
val_dyn_metrics = evaluate_predictions(val_labels, val_dyn_probs)

print("Running Dynamic Weight Selection on Test Set (One-Time Final Run)...")
test_dyn_probs, test_dyn_weights = dynamic_ensemble_predict(test_probs, candidate_weights, candidate_val_f1)

# Check weight variation across samples
weight_std_per_model = np.std(test_dyn_weights, axis=0)
print("\nProof of Input-Dependent Weight Variation (Std Dev of weights across test set):")
for name, s in zip(MODEL_NAMES, weight_std_per_model):
    print(f"  {name:15s} weight std dev = {s:.4f}")

assert np.any(weight_std_per_model > 1e-4), "Warning: Dynamic weights are static across samples!"
print("Verified: Ensemble weights are genuinely input-dependent and vary across X-ray images.")


In [ ]:
# Cell 8: Final Comprehensive Comparison on Held-Out Test Set
# Evaluation of:
# A. Individual Models (ResNet50, DenseNet121, EfficientNet-B3, Swin-T, ViT-B/16)
# B. Static Ensemble (Equal Weights: 0.20 across all 5)
# C. Best Fixed Validation Weight Ensemble
# D. Dynamic Input-Dependent Ensemble

comparison_records = []

# A. Individual Models
for idx, m_name in enumerate(MODEL_NAMES):
    metrics = evaluate_predictions(test_labels, test_probs[:, idx])
    comparison_records.append({
        "Model / Ensemble Configuration": f"Individual: {m_name}",
        "Accuracy (%)": f"{metrics['Accuracy']*100:.2f}%",
        "Precision": f"{metrics['Precision']:.4f}",
        "Recall": f"{metrics['Recall']:.4f}",
        "F1-Score": f"{metrics['F1']:.4f}",
        "ROC-AUC": f"{metrics['AUC']:.4f}",
        "Optimal Thr": f"{metrics['Threshold']:.3f}"
    })

# B. Static Equal-Weights Ensemble
equal_w = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
test_equal_probs = np.dot(test_probs, equal_w)
metrics_equal = evaluate_predictions(test_labels, test_equal_probs)
comparison_records.append({
    "Model / Ensemble Configuration": "Static Ensemble (Equal Weights 0.20)",
    "Accuracy (%)": f"{metrics_equal['Accuracy']*100:.2f}%",
    "Precision": f"{metrics_equal['Precision']:.4f}",
    "Recall": f"{metrics_equal['Recall']:.4f}",
    "F1-Score": f"{metrics_equal['F1']:.4f}",
    "ROC-AUC": f"{metrics_equal['AUC']:.4f}",
    "Optimal Thr": f"{metrics_equal['Threshold']:.3f}"
})

# C. Best Fixed Validation-Weight Ensemble
test_best_fixed_probs = np.dot(test_probs, best_fixed_weights)
metrics_fixed = evaluate_predictions(test_labels, test_best_fixed_probs)
comparison_records.append({
    "Model / Ensemble Configuration": f"Best Fixed Val Weights Ensemble",
    "Accuracy (%)": f"{metrics_fixed['Accuracy']*100:.2f}%",
    "Precision": f"{metrics_fixed['Precision']:.4f}",
    "Recall": f"{metrics_fixed['Recall']:.4f}",
    "F1-Score": f"{metrics_fixed['F1']:.4f}",
    "ROC-AUC": f"{metrics_fixed['AUC']:.4f}",
    "Optimal Thr": f"{metrics_fixed['Threshold']:.3f}"
})

# D. Dynamic Input-Dependent Ensemble
# Note: Threshold selected on validation set to strictly prevent test leakage
val_opt_thr = val_dyn_metrics["Threshold"]
test_dyn_preds = (test_dyn_probs >= val_opt_thr).astype(int)
dyn_acc = accuracy_score(test_labels, test_dyn_preds)
dyn_prec = precision_score(test_labels, test_dyn_preds, zero_division=0)
dyn_rec = recall_score(test_labels, test_dyn_preds, zero_division=0)
dyn_f1 = f1_score(test_labels, test_dyn_preds, zero_division=0)
dyn_auc = roc_auc_score(test_labels, test_dyn_probs)

comparison_records.append({
    "Model / Ensemble Configuration": "Dynamic Ensemble (Input-Dependent Weights)",
    "Accuracy (%)": f"{dyn_acc*100:.2f}%",
    "Precision": f"{dyn_prec:.4f}",
    "Recall": f"{dyn_rec:.4f}",
    "F1-Score": f"{dyn_f1:.4f}",
    "ROC-AUC": f"{dyn_auc:.4f}",
    "Optimal Thr": f"{val_opt_thr:.3f}"
})

df_comparison = pd.DataFrame(comparison_records)
print("\n" + "="*80)
print("FINAL TEST SET PERFORMANCE COMPARISON (INDIVIDUAL VS STATIC VS DYNAMIC)")
print("="*80)
print(df_comparison.to_markdown(index=False))
print("="*80)

print("\nClassification Report for Final Dynamic Ensemble on Test Set:")
print(classification_report(test_labels, test_dyn_preds, target_names=["NORMAL", "PNEUMONIA"], digits=4))


In [ ]:
# Cell 9: Per-Sample Dynamic Weight Observation Export (CSV)
df_samples = pd.DataFrame({
    "sample_index": np.arange(len(test_labels)),
    "actual_label": test_labels,
    "label_name": ["PNEUMONIA" if y == 1 else "NORMAL" for y in test_labels],
    "resnet_probability": np.round(test_probs[:, 0], 4),
    "densenet_probability": np.round(test_probs[:, 1], 4),
    "efficientnet_probability": np.round(test_probs[:, 2], 4),
    "swin_probability": np.round(test_probs[:, 3], 4),
    "vit_probability": np.round(test_probs[:, 4], 4),
    "resnet_weight": np.round(test_dyn_weights[:, 0], 4),
    "densenet_weight": np.round(test_dyn_weights[:, 1], 4),
    "efficientnet_weight": np.round(test_dyn_weights[:, 2], 4),
    "swin_weight": np.round(test_dyn_weights[:, 3], 4),
    "vit_weight": np.round(test_dyn_weights[:, 4], 4),
    "weight_sum": np.round(np.sum(test_dyn_weights, axis=1), 4),
    "final_probability": np.round(test_dyn_probs, 4),
    "final_prediction": (test_dyn_probs >= val_dyn_metrics["Threshold"]).astype(int)
})

csv_export_path = os.path.join(OUTPUT_DIR, "dynamic_ensemble_sample_weights.csv")
df_samples.to_csv(csv_export_path, index=False)
print(f"Per-sample dynamic weights and probabilities saved to: {csv_export_path}")

print("\nSample Rows Showing Genuine Input-Dependent Weight Variations:")
print(df_samples[["actual_label", "resnet_weight", "densenet_weight", "efficientnet_weight", "swin_weight", "vit_weight", "final_probability"]].head(10).to_string())


In [ ]:
# Cell 10: Dynamic Calibrator (Preserved Post-Hoc Calibration Adapted for 5 Models)
class DynamicCalibrator5Model(nn.Module):
    """
    Post-hoc temperature scaling based on the variance across all 5 model probabilities.
    Input: Variance across the 5 models
    Output: Calibrated probability
    """
    def __init__(self):
        super().__init__()
        self.temp_net = nn.Sequential(
            nn.Linear(1, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Softplus() # strictly positive temperature
        )
        for m in self.temp_net.modules():
            if isinstance(m, nn.Linear):
                nn.init.uniform_(m.weight, -0.1, 0.1)
                nn.init.constant_(m.bias, 0.5)

    def forward(self, ensemble_logits, prob_variance):
        temp = self.temp_net(prob_variance) + 1e-6
        calib_logits = ensemble_logits / temp
        return calib_logits, temp

# Compute logit variance across 5 models
val_prob_var = torch.tensor(np.var(val_probs, axis=1, keepdims=True), dtype=torch.float32).to(DEVICE)
test_prob_var = torch.tensor(np.var(test_probs, axis=1, keepdims=True), dtype=torch.float32).to(DEVICE)

# Convert dynamic ensemble probabilities to logit domain for calibration
eps = 1e-6
val_dyn_logits = torch.tensor(np.log(np.clip(val_dyn_probs, eps, 1 - eps) / (1 - np.clip(val_dyn_probs, eps, 1 - eps))), dtype=torch.float32).unsqueeze(1).to(DEVICE)
test_dyn_logits = torch.tensor(np.log(np.clip(test_dyn_probs, eps, 1 - eps) / (1 - np.clip(test_dyn_probs, eps, 1 - eps))), dtype=torch.float32).unsqueeze(1).to(DEVICE)

val_labels_t = torch.tensor(val_labels, dtype=torch.float32).unsqueeze(1).to(DEVICE)

calibrator = DynamicCalibrator5Model().to(DEVICE)
optimizer_calib = optim.Adam(calibrator.parameters(), lr=1e-3, weight_decay=1e-4)
criterion_calib = nn.BCEWithLogitsLoss()

print("Training Dynamic Calibrator on Validation set...")
calibrator.train()
for epoch in range(15):
    optimizer_calib.zero_grad()
    cal_logits, _ = calibrator(val_dyn_logits, val_prob_var)
    loss = criterion_calib(cal_logits, val_labels_t)
    loss.backward()
    optimizer_calib.step()
    if (epoch + 1) % 5 == 0:
        print(f"Calibrator Epoch {epoch+1}/15 - Loss: {loss.item():.4f}")

calibrator.eval()
with torch.no_grad():
    test_cal_logits, test_temps = calibrator(test_dyn_logits, test_prob_var)
    test_cal_probs = torch.sigmoid(test_cal_logits).cpu().numpy().flatten()

cal_metrics = evaluate_predictions(test_labels, test_cal_probs)
print(f"\nCalibrated Dynamic Ensemble -> Test AUC: {cal_metrics['AUC']:.4f} | Test F1: {cal_metrics['F1']:.4f}")

# Save calibrator checkpoint
calib_save_path = os.path.join(OUTPUT_DIR, "best_dynamic_calibrator_5model.pth")
torch.save(calibrator.state_dict(), calib_save_path)
print(f"Saved Calibrator model to: {calib_save_path}")


In [ ]:
# Cell 11: Scientific Visualizations
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig = plt.figure(figsize=(18, 12))

# 1. Distribution of selected dynamic weights per model
ax1 = plt.subplot(2, 2, 1)
df_weights = pd.DataFrame(test_dyn_weights, columns=MODEL_NAMES)
df_weights.boxplot(ax=ax1, patch_artist=True, boxprops=dict(facecolor='lightblue', color='blue'))
ax1.set_title("Distribution of Selected Dynamic Weights Across Test Set", fontsize=12, fontweight='bold')
ax1.set_ylabel("Assigned Weight (Sum = 1.0)", fontsize=11)
ax1.set_ylim(-0.02, 0.7)

# 2. Dynamic weights across 10 diverse test samples
ax2 = plt.subplot(2, 2, 2)
df_sample_weights = pd.DataFrame(test_dyn_weights[:10], columns=MODEL_NAMES)
df_sample_weights.plot(kind='bar', stacked=True, ax=ax2, colormap='tab10')
ax2.set_title("Input-Dependent Weights Across 10 Representative X-ray Samples", fontsize=12, fontweight='bold')
ax2.set_xlabel("Sample Index", fontsize=11)
ax2.set_ylabel("Weight Proportion", fontsize=11)
ax2.legend(loc='upper right', bbox_to_anchor=(1.25, 1.0))
ax2.set_ylim(0, 1.05)

# 3. ROC Curves Comparison
ax3 = plt.subplot(2, 2, 3)
fpr_eq, tpr_eq, _ = roc_curve(test_labels, test_equal_probs)
fpr_fix, tpr_fix, _ = roc_curve(test_labels, test_best_fixed_probs)
fpr_dyn, tpr_dyn, _ = roc_curve(test_labels, test_dyn_probs)
fpr_cal, tpr_cal, _ = roc_curve(test_labels, test_cal_probs)

ax3.plot(fpr_eq, tpr_eq, label=f"Static Equal Weights (AUC = {metrics_equal['AUC']:.4f})", linestyle=':')
ax3.plot(fpr_fix, tpr_fix, label=f"Best Fixed Val Weights (AUC = {metrics_fixed['AUC']:.4f})", linestyle='--')
ax3.plot(fpr_dyn, tpr_dyn, label=f"Dynamic Weighted Ensemble (AUC = {dyn_auc:.4f})", linewidth=2.5, color='darkgreen')
ax3.plot(fpr_cal, tpr_cal, label=f"Dynamic + Calibrated (AUC = {cal_metrics['AUC']:.4f})", linewidth=1.5, color='purple')
ax3.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax3.set_title("ROC Curve: Static vs. Dynamic Weighted Ensemble", fontsize=12, fontweight='bold')
ax3.set_xlabel("False Positive Rate", fontsize=11)
ax3.set_ylabel("True Positive Rate", fontsize=11)
ax3.legend(loc="lower right")

# 4. Metric Comparison Bar Chart
ax4 = plt.subplot(2, 2, 4)
configs = ["Equal Weights", "Best Fixed", "Dynamic", "Dyn + Calib"]
f1_scores_list = [metrics_equal['F1'], metrics_fixed['F1'], dyn_f1, cal_metrics['F1']]
auc_scores_list = [metrics_equal['AUC'], metrics_fixed['AUC'], dyn_auc, cal_metrics['AUC']]

x = np.arange(len(configs))
width = 0.35
ax4.bar(x - width/2, f1_scores_list, width, label='F1-Score', color='royalblue')
ax4.bar(x + width/2, auc_scores_list, width, label='ROC-AUC', color='coral')
ax4.set_xticks(x)
ax4.set_xticklabels(configs, fontsize=10)
ax4.set_ylim(0.7, 1.0)
ax4.set_title("Performance Comparison Across Ensemble Configurations", fontsize=12, fontweight='bold')
ax4.set_ylabel("Score", fontsize=11)
ax4.legend(loc='lower left')

plt.tight_layout()
viz_path = os.path.join(OUTPUT_DIR, "ensemble_comparison_plots.png")
plt.savefig(viz_path, dpi=300)
plt.show()
print(f"Comprehensive visualizations saved to: {viz_path}")
